## Initializing the NUTS-HMC sampler

In this example, we show how to use parameter estimates returned by any of Stan's inference algorithms as initial values for Stan's NUTS-HMC sampler. These include:

* [Pathfinder ](https://mc-stan.org/docs/cmdstan-guide/pathfinder-config.html) 
* [ADVI ](https://mc-stan.org/docs/cmdstan-guide/variational_config.html) 
* [Laplace](https://mc-stan.org/docs/cmdstan-guide/laplace_sample_config.html)
* [Optimization](https://mc-stan.org/docs/cmdstan-guide/optimize_config.html)
* [NUTS-HMC MCMC](https://mc-stan.org/docs/cmdstan-guide/mcmc_config.html)

By default, the NUTS-HMC sampler randomly initializes all (unconstrained) model parameters uniformly in the interval (-2, 2).  If this interval is far from the typical set of the posterior, initializing sampling from these approximation algorithms can speed up and improve adaptation.

### Model and data

The Stan model and data are taken from the [posteriordb package](https://github.com/stan-dev/posteriordb).

We use the [blr model](https://github.com/stan-dev/posteriordb/blob/master/posterior_database/models/stan/blr.stan),
a Bayesian standard linear regression model with noninformative priors,
and its corresponding simulated dataset [sblri.json](https://github.com/stan-dev/posteriordb/blob/master/posterior_database/data/data/sblri.json.zip),
which was simulated via script [sblr.R](https://github.com/stan-dev/posteriordb/blob/master/posterior_database/data/data-raw/sblr/sblr.R).
For convenience, this example assumes the posteriordb model and data are local, in files `blr.stan` and `sblri.json`.

In [ ]:
import os
from cmdstanpy import CmdStanModel

stan_file = 'blr.stan' # basic linear regression
data_file = 'sblri.json' # simulated data

model = CmdStanModel(stan_file=stan_file)

print(model.code())

### Demonstration with Stan's `pathfinder` method

Initializing the sampler with estimates from any previous inference algorithm follows the same general usage pattern. First, we call the 
corresponding method on the `CmdStanModel` object. From the resulting fit, we call the `.create_inits()` 
method to construct a set of per-chain initializations for the model parameters. To make it explicit, 
we will walk through the process using the `pathfinder` method (which wraps the 
CmdStan [pathfinder ](https://mc-stan.org/docs/cmdstan-guide/pathfinder-config.html) method).

Pathfinder locates normal approximations to the target
density along a quasi-Newton optimization path, with local covariance
estimated using the negative inverse Hessian estimates produced by the
LBFGS optimizer. Pathfinder returns draws from the Gaussian approximation
with the lowest estimated Kullback-Leibler (KL) divergence to the true
posterior.
By default, CmdStanPy runs multi-path Pathfinder which returns an importance-resampled 
set of draws over the outputs of 4 independent single-path Pathfinders.
This better matches non-normal target densities and also mitigates
the problem of L-BFGS getting stuck at local optima or in saddle points on plateaus.

We obtain Pathfinder estimates by calling the `.pathfinder()` method which returns a `CmdStanPathfinder` object:

In [ ]:
pathfinder_fit = model.pathfinder(data=data_file, seed=123)

Posteriordb provides reference posteriors for all models. For the blr model, conditioned on the dataset `sblri.json`, the reference posteriors can be found in the [sblri-blr.json](https://github.com/stan-dev/posteriordb/blob/master/posterior_database/reference_posteriors/summary_statistics/mean/mean/sblri-blr.json) file.

The reference posteriors for all elements of `beta` and `sigma` are all very close to 1.0.

The experiments reported in Figure 3 of the paper [Pathfinder: Parallel quasi-Newton variational inference](https://arxiv.org/abs/2108.03782) by Zhang et al. show that Pathfinder provides a better estimate of the posterior, as measured by the 1-Wasserstein distance to the reference posterior, than 75 iterations of the warmup Phase I algorithm used by the NUTS-HMC sampler.
Furthermore, Pathfinder is more computationally efficient, requiring fewer evaluations of the log density and gradient functions. Therefore, using the Pathfinder estimates to initialize the parameter values for the NUTS-HMC sampler can allow the sampler to do a better job of adapting the stepsize and metric during warmup, resulting in better performance and estimation.

We construct the parameter inits for full MCMC sampling below. The `.create_inits()` default behavior is to create inits for four chains to correspond with the sampling defaults. You can requests more or less by modifying the `chains` keyword argument.

In [ ]:
pathfinder_inits = pathfinder_fit.create_inits()
for chain_init in pathfinder_inits:
    print(chain_init)

We see that the Pathfinder inits are close the reference posteriors for the parameters. To use these inits, we pass the `pathfinder_inits` object to the `inits` kwarg:

In [ ]:
mcmc_pathfinder_inits_fit = model.sample(
    data=data_file, inits=pathfinder_inits, iter_warmup=75, seed=12345
)

In [ ]:
print(mcmc_pathfinder_inits_fit.diagnose())

Despite only running 75 warmup iterations, all posterior diagnostics from the sampler look good.

In [ ]:
mcmc_pathfinder_inits_fit.summary()

If we were to instead use the default random parameter initializations, we would need to run more warmup iterations to produce useful samples. For example, if we only run the same 75 warmup iterations with random inits, the result fails to estimate `sigma` correctly: 

In [ ]:
mcmc_random_inits_fit = model.sample(data=data_file, iter_warmup=75, seed=12345)

In [ ]:
mcmc_random_inits_fit.summary()

In [ ]:
print(mcmc_random_inits_fit.diagnose())

The diagnostics clearly indicate problems with estimating `sigma`. In this case, it is necessary to run the model with at least 150 warmup iterations to produce a good set of estimates when starting from the default initialization.

### Other inference algorithms

We can follow the same pattern with Stan's ADVI algorithm by first using the `CmdStanModel.variationl` method. Because this algorithm is unstable and may fail to converge, we run it with argument `require_converged` set to `False`.  We also specify a seed, to avoid instabilities as well as for reproducibility.

In [ ]:
vb_fit = model.variational(data=data_file, require_converged=False, seed=123)

The ADVI algorithm provides estimates of all model parameters.

The `variational` method returns a `CmdStanVB` object, which similarly can construct a set of inits with the `.create_inits()` method:

In [ ]:
vb_inits = vb_fit.create_inits()
for chain_init in vb_inits:
    print(chain_init)

Which can be passed to the `inits` keyword argument of the sample method.

In [ ]:
mcmc_vb_inits_fit = model.sample(
    data=data_file, inits=vb_inits, iter_warmup=75, seed=12345
)

In [ ]:
print(mcmc_vb_inits_fit.diagnose())

In [ ]:
mcmc_vb_inits_fit.summary()

The sampler estimates match the reference posterior with no diagnostic issues.

Inits can also be constructed from the `laplace` method:

In [ ]:
laplace_inits = model.laplace_sample(data=data_file, seed=123).create_inits()

In [ ]:
mcmc_laplace_inits_fit = model.sample(
    data=data_file, inits=laplace_inits, iter_warmup=75, seed=12345
)

In [ ]:
print(mcmc_laplace_inits_fit.diagnose())

And the `optimize` method. Since optimizations attempts to return a posterior mode, this will initialize all chains at the same point, which is not typically ideal.

In [ ]:
optimized_inits = model.optimize(data=data_file, seed=123).create_inits()

In [ ]:
mcmc_optimize_inits_fit = model.sample(
    data=data_file, inits=optimized_inits, iter_warmup=75, seed=12345
)

In [ ]:
print(mcmc_optimize_inits_fit.diagnose())

It is also possible to use the output of the `sample()` method itself to construct inits to be fed into a future sampling run:

In [ ]:
first_mcmc_inits = model.sample(
    data=data_file, iter_warmup=75, seed=12345
).create_inits()

In [ ]:
second_mcmc_fit = model.sample(data=data_file, inits=first_mcmc_inits, iter_warmup=75, seed=12345)

In [ ]:
print(second_mcmc_fit.diagnose())

We see that despite the initial sampling issues in the first MCMC run, the inits sourced from that run result in reasonable sampling in the second run.